This script focuses on **testing fairness**, not on training new models.  
Its main goal is to evaluate whether different models behave fairly across groups after they are trained.

The code runs two AIDE experiments: a baseline model and a fairness-aware model.  
After AIDE generates the model scripts, the script executes them and collects their predictions on the same test data.  
Fairness is then checked by comparing model outputs across groups, using group-based metrics such as accuracy, F1 score, and differences between protected and non-protected groups (for example, differences in true positive rates).

To make the evaluation reliable, the script ensures that the dataset is always found, that target labels are detected automatically, and that the same test data is used for all models.  
The results are saved as CSV files so that the fairness of different models can be compared in a clear and reproducible way.

**References :**
- Hardt et al. (2016). *Equality of Opportunity in Supervised Learning*. NeurIPS.  
- Barocas & Selbst (2016). *Big Data’s Disparate Impact*. California Law Review.  
- Pedregosa et al. (2011). *Scikit-learn: Machine Learning in Python*.


In [ ]:
from __future__ import annotations

import os
import sys
import shutil
import subprocess
import re
from pathlib import Path
from typing import List, Optional

import pandas as pd


# =================================================
# HARD REQUIREMENT: must use 'aideml' distribution
# =================================================
def require_aideml_distribution() -> str:
    """
    Verify the pip distribution 'aideml' is installed in THIS environment.
    """
    try:
        import importlib.metadata as md
        return md.version("aideml")
    except Exception as e:
        raise RuntimeError(
            "Requirement not satisfied: distribution 'aideml' is not installed in this environment.\n"
            "Install in THIS kernel/env with:\n"
            f"  {sys.executable} -m pip install -U aideml\n"
            "or (in Jupyter):\n"
            "  %pip install -U aideml\n"
        ) from e


def require_aide_importable() -> None:
    """
    Optional sanity check: after installing 'aideml', the importable module 'aide' should exist.
    """
    try:
        import aide  # type: ignore  # noqa: F401
    except Exception as e:
        raise RuntimeError(
            "The 'aide' module is not importable even though 'aideml' is installed.\n"
            "Try reinstalling:\n"
            f"  {sys.executable} -m pip install -U --force-reinstall aideml\n"
        ) from e


def aide_entry_cmd() -> List[str]:
    """
    IMPORTANT (Windows):
    - `python -m aide` fails because 'aide' has no __main__
    - Must use the CLI executable 'aide' (aide.exe)
    """
    exe = shutil.which("aide")
    if exe:
        return [exe]

    candidate = Path(sys.executable).parent / "aide.exe"
    if candidate.exists():
        return [str(candidate)]

    raise RuntimeError(
        "Cannot find AIDE executable 'aide' in this environment.\n"
        f"Install: {sys.executable} -m pip install -U aideml\n"
    )


# =================================================
# Utilities
# =================================================
def find_data_file(start: Path) -> Path:
    """
    Find any dataset file whose stem contains 'Modified_Churn_Modelling' (case-insensitive).
    """
    exts = ("*.csv", "*.xlsx", "*.xls")
    for ext in exts:
        for p in start.rglob(ext):
            if "modified_churn_modelling" in p.stem.lower():
                return p
    raise FileNotFoundError(f"No file containing 'Modified_Churn_Modelling' found under {start}")


def precheck_dataset_readable(dataset: Path) -> None:
    """
    Fail fast if pandas can't read the dataset.
    """
    if dataset.suffix.lower() == ".csv":
        pd.read_csv(dataset, nrows=5)
    else:
        pd.read_excel(dataset, nrows=5)


def to_posix(p: Path) -> str:
    return str(p).replace("\\", "/")


def tail_text(path: Path, max_lines: int = 200) -> str:
    if not path.exists():
        return "<log file not found>"
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    return "\n".join(lines[-max_lines:] if len(lines) > max_lines else lines)


def run_cmd_stream(cmd: List[str], log_path: Path, cwd: Optional[Path] = None) -> int:
    """
    Run a command, stream stdout/stderr, and save to log.
    """
    print("\n[CMD]")
    print(" ".join(cmd))
    print(f"[CWD] {cwd}")
    print(f"[LOG] {log_path}")

    env = os.environ.copy()
    env["PYTHONUTF8"] = "1"
    env.setdefault("PYTHONIOENCODING", "utf-8")

    log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open("w", encoding="utf-8", errors="replace") as f:
        p = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            cwd=str(cwd) if cwd else None,
            env=env,
            text=False,
        )
        assert p.stdout is not None

        for b in iter(lambda: p.stdout.readline(), b""):
            line = b.decode("utf-8", errors="replace")
            print(line, end="")
            f.write(line)

        ret = p.wait()
        f.write(f"\n[EXIT_CODE] {ret}\n")

    return ret


def pick_generated_script(workspace_dir: Path) -> Path:
    """
    Prefer solution.py if it exists; otherwise pick newest .py under workspace.
    """
    sols = list(workspace_dir.rglob("solution.py"))
    if sols:
        sols.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        return sols[0]

    pys = [p for p in workspace_dir.rglob("*.py") if p.is_file()]
    if pys:
        pys.sort(key=lambda p: p.stat().st_mtime, reverse=True)
        return pys[0]

    raise FileNotFoundError(f"No generated Python file found under workspace: {workspace_dir}")


# =================================================
# Dataset placement (handle scripts that chdir to ./working)
# =================================================
def copy_dataset_with_canonical_name(src: Path, dst_dir: Path) -> None:
    """
    Copy dataset to dst_dir with:
    - its original name
    - a canonical name that scripts may hardcode
    """
    dst_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src, dst_dir / src.name)

    if src.suffix.lower() == ".csv":
        shutil.copy2(src, dst_dir / "Modified_Churn_Modelling.csv")
    else:
        shutil.copy2(src, dst_dir / "Modified_Churn_Modelling.xlsx")


def place_dataset_multiple_locations(dataset_src: Path, run_dir: Path) -> None:
    """
    Put the dataset in multiple likely working folders.
    NOTE: include 'input' because some generated scripts hardcode ./input/...
    """
    copy_dataset_with_canonical_name(dataset_src, run_dir)
    for sub in ("working", "work", "data", "dataset", "outputs", "output", "input"):
        copy_dataset_with_canonical_name(dataset_src, run_dir / sub)


# =================================================
# PATCH: dataset finder + target selection + OHE compat
# + fix hardcoded pd.read_csv("./input/...Modified_Churn_Modelling...")
# =================================================
_PATCH_UNIFIED = r'''
# ===========================
# PATCHED HELPERS (injected by runner)
# ===========================
from pathlib import Path
import sklearn.preprocessing as _sk_pre

def _patched_find_dataset():
    exts = (".csv", ".xlsx", ".xls")
    roots = [Path("."), Path("..")]

    preferred = []
    any_files = []

    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if p.is_file() and p.suffix.lower() in exts:
                any_files.append(p)
                if "modified_churn_modelling" in p.stem.lower():
                    preferred.append(p)

    if preferred:
        preferred.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        return preferred[0]

    if any_files:
        any_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        return any_files[0]

    raise FileNotFoundError("No dataset found")

def _patched_pick_target_column(df):
    candidates = ["Exited", "target", "y", "label", "value", "outcome", "decision"]
    cols_lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in cols_lower:
            return cols_lower[c.lower()]
    return df.columns[-1]

def _patched_make_ohe(**kwargs):
    # Robust for sklearn>=1.2 (sparse_output) and sklearn<1.2 (sparse)
    OHE = getattr(_sk_pre, "OneHotEncoder")
    try:
        return OHE(sparse_output=False, **kwargs)
    except TypeError:
        return OHE(sparse=False, **kwargs)
'''


def patch_generated_script(script_path: Path) -> None:
    """
    Patch generated AIDE script to be robust:
    - dataset discovery via _patched_find_dataset
    - target selection via _patched_pick_target_column
    - OneHotEncoder compatibility across sklearn versions
    - replace hardcoded read_csv/read_excel paths containing Modified_Churn_Modelling
    """
    txt = script_path.read_text(encoding="utf-8", errors="replace")

    already_has_patch = (
        "_patched_find_dataset" in txt
        and "_patched_pick_target_column" in txt
        and "_patched_make_ohe" in txt
    )

    # 1) Inject patch block near the top (after __future__ imports)
    lines = txt.splitlines()
    if not already_has_patch:
        insert_at = 0
        for i, line in enumerate(lines[:200]):
            if line.startswith("from __future__"):
                insert_at = i + 1
        lines[insert_at:insert_at] = _PATCH_UNIFIED.strip("\n").splitlines()

    txt2 = "\n".join(lines) + "\n"

    # 2) Redirect common dataset lookup function calls to _patched_find_dataset()
    txt2 = re.sub(r"\bfind_dataset\s*\(\s*\)", "_patched_find_dataset()", txt2)
    txt2 = re.sub(r"\blocate_dataset\s*\(\s*\)", "_patched_find_dataset()", txt2)
    txt2 = re.sub(r"\bfind_data\s*\(\s*\)", "_patched_find_dataset()", txt2)

    # 2.5) Replace hardcoded dataset paths in pd.read_csv/read_excel that mention Modified_Churn_Modelling
    # e.g. pd.read_csv("./input/Modified_Churn_Modelling - Modified_Churn_Modelling.csv")
    txt2 = re.sub(
        r"pd\.read_csv\s*\(\s*([rR]?[\"'][^\"']*modified[_\-\s]*churn[_\-\s]*modelling[^\"']*[\"'])",
        "pd.read_csv(_patched_find_dataset())",
        txt2,
        flags=re.I,
    )
    txt2 = re.sub(
        r"pd\.read_excel\s*\(\s*([rR]?[\"'][^\"']*modified[_\-\s]*churn[_\-\s]*modelling[^\"']*[\"'])",
        "pd.read_excel(_patched_find_dataset())",
        txt2,
        flags=re.I,
    )

    # 3) Replace hardcoded ["Exited"] usage with [target_col]
    txt2 = re.sub(r'\[\s*["\']Exited["\']\s*\]', "[target_col]", txt2)

    # 4) Replace OneHotEncoder(...) with _patched_make_ohe(...)
    # IMPORTANT: avoid touching attribute access like _sk_pre.OneHotEncoder(...)
    txt2 = re.sub(r"(?<!\.)\bOneHotEncoder\s*\(", "_patched_make_ohe(", txt2)

    # Remove sparse/sparse_output arguments in patched calls (helper sets them)
    txt2 = re.sub(
        r"\b_patched_make_ohe\s*\(\s*([^)]*?)\bsparse_output\s*=\s*(True|False)\s*,?\s*",
        r"_patched_make_ohe(\1",
        txt2,
    )
    txt2 = re.sub(
        r"\b_patched_make_ohe\s*\(\s*([^)]*?)\bsparse\s*=\s*(True|False)\s*,?\s*",
        r"_patched_make_ohe(\1",
        txt2,
    )

    # 5) Insert `target_col = _patched_pick_target_column(<dfvar>)` right after first load line
    txt_lines = txt2.splitlines()
    out_lines: List[str] = []
    inserted = False

    load_pat = re.compile(r"^(\s*)(raw_df|df)\s*=\s*.*pd\.read_(csv|excel)\b", re.I)

    for line in txt_lines:
        out_lines.append(line)
        m = load_pat.search(line)
        if (not inserted) and m:
            indent = m.group(1)
            var = m.group(2)
            out_lines.append(f"{indent}target_col = _patched_pick_target_column({var})")
            inserted = True

    # looser fallback
    if not inserted:
        out2: List[str] = []
        loose_pat = re.compile(r"^(\s*)(raw_df|df)\s*=\s*.*read_(csv|excel)\b", re.I)
        for line in out_lines:
            out2.append(line)
            m = loose_pat.search(line)
            if (not inserted) and m:
                indent = m.group(1)
                var = m.group(2)
                out2.append(f"{indent}target_col = _patched_pick_target_column({var})")
                inserted = True
        out_lines = out2

    # final fallback
    if not inserted:
        out_lines.insert(0, "target_col = None  # injected fallback (runner could not detect df load line)")

    script_path.write_text("\n".join(out_lines) + "\n", encoding="utf-8")


# =================================================
# Execute generated scripts
# =================================================
def run_solution(script: Path, run_dir: Path, dataset_src: Path, tag: str) -> Path:
    """
    Copy, patch, and execute the generated script; then locate predictions.csv.
    Also copies predictions to a tag-specific filename for easier comparison.
    """
    run_dir.mkdir(parents=True, exist_ok=True)

    local_script = run_dir / script.name
    shutil.copy2(script, local_script)

    # Make dataset discoverable even if the script does os.chdir("working")
    place_dataset_multiple_locations(dataset_src, run_dir)

    # Patch generated script
    patch_generated_script(local_script)

    log = run_dir / "run.log"
    ret = run_cmd_stream([sys.executable, local_script.name], log, cwd=run_dir)

    if ret != 0:
        raise RuntimeError(
            f"Generated script failed with exit code {ret}.\n"
            f"Log: {log}\n\n"
            f"========== LOG TAIL ==========\n{tail_text(log, 200)}\n"
            f"========== END LOG TAIL =========="
        )

    preds = list(run_dir.rglob("predictions.csv"))
    if not preds:
        raise FileNotFoundError(
            "predictions.csv not produced.\n"
            f"Run dir: {run_dir}\n"
            f"Log: {log}\n\n"
            f"========== LOG TAIL ==========\n{tail_text(log, 200)}\n"
            f"========== END LOG TAIL =========="
        )

    preds.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    pred_path = preds[0]

    tagged = run_dir / f"predictions_{tag}.csv"
    shutil.copy2(pred_path, tagged)
    return tagged


# =================================================
# Prompts
# =================================================
BASELINE_PROMPT = """
Generate a SINGLE runnable Python script named solution.py.

Context:
This is a BANK customer decision / credit-scoring style binary classification task.

Data:
- A file named Modified_Churn_Modelling.csv OR Modified_Churn_Modelling.xlsx
  will be present under the working directory.
- The script MUST locate the dataset by recursively searching from
  the current directory.
- If the file is Excel, load it with pandas.read_excel;
  if CSV, use pandas.read_csv.

Target definition (binary classification):
- Determine the target column as follows:
  1) If any of these columns exist, use the first found:
     Exited, target, y, label, value, outcome, decision
  2) Otherwise, use the LAST column
- Treat the POSITIVE class as the target outcome.

Modeling (traditional baseline):
- Pipeline:
  - categorical: OneHotEncoder(handle_unknown="ignore")
  - numerical: StandardScaler
  - ColumnTransformer
- Classifier: LogisticRegression(max_iter=2000)
- Train-test split: test_size=0.2, random_state=42, stratify=y

Evaluation:
- Compute and print:
  - Accuracy
  - Balanced Accuracy
  - F1-score (binary)
- Print results EXACTLY:
  VAL_ACC=<float>
  VAL_BACC=<float>
  VAL_F1=<float>

Artifacts:
- Save predictions.csv containing:
  - y_true
  - y_pred
  - all original feature columns
- Save model.joblib

Prediction API:
- Implement predict_csv(input_path, output_path)

Constraints:
- Use ONLY: numpy, pandas, scikit-learn, joblib
- English comments ONLY
- The script must run end-to-end without manual edits.
""".strip()
# References:
# - ESL (Hastie et al., 2009)
# - ISLR (James et al., 2021)
# - Credit Scoring Review (Hand & Henley, 1997)
# - scikit-learn documentation


FAIRNESS_PROMPT = """
Generate a SINGLE runnable Python script named solution.py.

Context:
This experiment follows the standard experimental setting of
fairness-aware machine learning for CREDIT SCORING as described
in recent academic literature.
The task is a BANK customer decision problem with potential bias
with respect to protected attributes (e.g., gender).

Data:
- A file named Modified_Churn_Modelling.csv OR Modified_Churn_Modelling.xlsx
  will be present under the working directory.
- The script MUST locate the dataset by recursively searching from
  the current directory.
- If the file is Excel, load it with pandas.read_excel;
  if CSV, use pandas.read_csv.

Target definition (binary classification):
- Determine the target column as follows:
  1) If any of these columns exist, use the first found:
     Exited, target, y, label, value, outcome, decision
  2) Otherwise, use the LAST column
- Treat the POSITIVE class as the target outcome.

Protected attributes (fairness requirement):
- If columns such as Gender, Sex, Age, Geography exist,
  treat them as protected attributes.
- Gender (or Sex) should be considered the primary protected attribute
  if present.
- Do NOT remove protected attributes.

Modeling (traditional baseline):
- Same pipeline as baseline:
  OneHotEncoder(handle_unknown="ignore") + StandardScaler + LogisticRegression(max_iter=2000)
- Train-test split: test_size=0.2, random_state=42, stratify=y

Evaluation:
- Compute and print:
  - Accuracy
  - Balanced Accuracy
  - F1-score (binary)
- Print results EXACTLY:
  VAL_ACC=<float>
  VAL_BACC=<float>
  VAL_F1=<float>

Fairness evaluation (group fairness, simplified):
- If a protected attribute exists:
  - Statistical Parity difference:
    P(y_pred=1 | protected) - P(y_pred=1 | non-protected)
  - Equal Opportunity difference:
    difference in True Positive Rate between groups
- Print these values for analysis purposes only.

Artifacts:
- Save predictions.csv containing:
  - y_true
  - y_pred
  - all original feature columns
- Save model.joblib

Prediction API:
- Implement predict_csv(input_path, output_path)

Constraints:
- Use ONLY: numpy, pandas, scikit-learn, joblib
- No external fairness libraries
- English comments ONLY
- The script must run end-to-end without manual edits.
""".strip()
# References:
# - Hardt, M., Price, E., Srebro, N. (2016).
#   Equality of Opportunity in Supervised Learning.
# - Barocas, S., Selbst, A. D. (2016).
#   Big Data's Disparate Impact. California Law Review.
# - Kleinberg, J., Mullainathan, S., Raghavan, M. (2017).
#   Inherent Trade-Offs in the Fair Determination of Risk Scores.
# - Fuster, A., Goldsmith-Pinkham, P., Ramadorai, T., Walther, A. (2019).
#   Predictably Unequal? The Effects of Machine Learning on Credit Markets.
# - Pedregosa, F., et al. (2011).
#   Scikit-learn: Machine Learning in Python.


# =================================================
# Main
# =================================================
def main() -> None:
    aideml_ver = require_aideml_distribution()
    require_aide_importable()

    print(f"[OK] aideml distribution: {aideml_ver}")
    print(f"[OK] python executable: {sys.executable}")
    print(f"[OK] aide executable: {shutil.which('aide')}")
    print(f"[OK] running AIDE via: {' '.join(aide_entry_cmd())}")

    project_root = Path(r"C:\TUB\RDEP")
    dataset = find_data_file(project_root)
    print(f"[FOUND DATASET] {dataset}")
    precheck_dataset_readable(dataset)

    base = Path.home() / "Documents"
    data_dir = base / "AIDE_BANKBIAS_INPUT"
    ws_root = base / "AIDE_BANKBIAS_WORKSPACE_NAMED"
    out_dir = ws_root / "OUTPUT"

    wsA = ws_root / "WS_BASELINE_RUN"
    wsB = ws_root / "WS_PROMPT_RUN"

    for d in (data_dir, wsA, wsB, out_dir):
        d.mkdir(parents=True, exist_ok=True)

    for p in data_dir.glob("*"):
        if p.is_file():
            p.unlink()
    shutil.copy2(dataset, data_dir / dataset.name)

    (wsA / "desc.txt").write_text(BASELINE_PROMPT, encoding="utf-8")
    (wsB / "desc.txt").write_text(FAIRNESS_PROMPT, encoding="utf-8")

    print("\n=== AIDE BASELINE ===")
    logA = wsA / "aide.log"
    retA = run_cmd_stream(
        [
            *aide_entry_cmd(),
            f"data_dir={to_posix(data_dir)}",
            f"workspace_dir={to_posix(wsA)}",
            f"desc_file={to_posix(wsA / 'desc.txt')}",
            "agent.steps=5",
            "agent.k_fold_validation=1",
            "generate_report=False",
            "copy_data=True",
        ],
        logA,
        cwd=wsA,
    )
    if retA != 0:
        raise RuntimeError(f"AIDE baseline run failed.\n\n{tail_text(logA, 200)}")

    genA = pick_generated_script(wsA)
    base_py = out_dir / "bankbias_baseline.py"
    shutil.copy2(genA, base_py)

    print("\n=== AIDE PROMPT (FAIRNESS PROMPT) ===")
    logB = wsB / "aide.log"
    retB = run_cmd_stream(
        [
            *aide_entry_cmd(),
            f"data_dir={to_posix(data_dir)}",
            f"workspace_dir={to_posix(wsB)}",
            f"desc_file={to_posix(wsB / 'desc.txt')}",
            "agent.steps=5",
            "agent.k_fold_validation=1",
            "generate_report=False",
            "copy_data=True",
        ],
        logB,
        cwd=wsB,
    )
    if retB != 0:
        raise RuntimeError(f"AIDE prompt run failed.\n\n{tail_text(logB, 200)}")

    genB = pick_generated_script(wsB)
    fair_py = out_dir / "bankbias_prompt.py"
    shutil.copy2(genB, fair_py)

    print("\n=== EXECUTE GENERATED SCRIPTS ===")
    predA = run_solution(base_py, out_dir / "RUN_BASELINE", dataset, tag="baseline")
    predB = run_solution(fair_py, out_dir / "RUN_PROMPT", dataset, tag="fairness")

    print("\n=== DONE ===")
    print("Baseline script:", base_py)
    print("Prompt script  :", fair_py)
    print("Baseline preds :", predA)
    print("Prompt preds   :", predB)


if __name__ == "__main__":
    main()


[OK] aideml distribution: 0.2.2
[OK] python executable: c:\Users\wenyi\dsenv\Scripts\python.exe
[OK] aide executable: c:\Users\wenyi\dsenv\Scripts\aide.EXE
[OK] running AIDE via: c:\Users\wenyi\dsenv\Scripts\aide.EXE
[FOUND DATASET] C:\TUB\RDEP\archive\Modified_Churn_Modelling - Modified_Churn_Modelling.csv

=== AIDE BASELINE ===

[CMD]
c:\Users\wenyi\dsenv\Scripts\aide.EXE data_dir=C:/Users/wenyi/Documents/AIDE_BANKBIAS_INPUT workspace_dir=C:/Users/wenyi/Documents/AIDE_BANKBIAS_WORKSPACE_NAMED/WS_BASELINE_RUN desc_file=C:/Users/wenyi/Documents/AIDE_BANKBIAS_WORKSPACE_NAMED/WS_BASELINE_RUN/desc.txt agent.steps=5 agent.k_fold_validation=1 generate_report=False copy_data=True
[CWD] C:\Users\wenyi\Documents\AIDE_BANKBIAS_WORKSPACE_NAMED\WS_BASELINE_RUN
[LOG] C:\Users\wenyi\Documents\AIDE_BANKBIAS_WORKSPACE_NAMED\WS_BASELINE_RUN\aide.log
⠋ Preparing agent workspace (copying and extracting files) ...
┌──────── AIDE is working on experiment: "2-venomous-inescapable-owl" ────────┐
│          

This script is designed to **test and compare fairness**, not to train models.  
It takes prediction results from two already trained models (a baseline model and a fairness-aware model) and evaluates how fair their decisions are with respect to a protected attribute.

First, the script loads the prediction files and automatically selects a protected attribute, such as Gender, Sex, Geography, or Age.  
If the attribute is numeric, it is split into two groups using the median. If it is categorical, the two largest groups are used.  
This makes the evaluation simple and consistent across different datasets.

Next, the script computes **overall performance metrics** (Accuracy, Balanced Accuracy, and F1) and **group-level statistics** for each protected group.  
These include the positive prediction rate, True Positive Rate (TPR), False Positive Rate (FPR), and confusion matrix counts.  
Based on these values, standard fairness metrics are calculated, such as Statistical Parity Difference (SPD), Equal Opportunity Difference (EOD), and Disparate Impact Ratio (DIR).

Finally, the results from the baseline model and the fairness-aware model are shown side by side.  
This allows a clear comparison of whether the fairness-oriented model reduces group gaps while keeping reasonable predictive performance.

**References :**
- Hardt, Price, & Srebro (2016). *Equality of Opportunity in Supervised Learning*. NeurIPS.  
- Barocas & Selbst (2016). *Big Data’s Disparate Impact*. California Law Review.  
- Kleinberg, Mullainathan, & Raghavan (2017). *Inherent Trade-Offs in the Fair Determination of Risk Scores*.  
- Fuster et al. (2019). *Predictably Unequal? The Effects of Machine Learning on Credit Markets*.


In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple


# =================================================
# Utility functions
# =================================================
def _safe_div(a: float, b: float) -> float:
    """Safe division, returns NaN if denominator is zero."""
    return float(a / b) if b not in (0.0, 0) else float("nan")


def compute_basic_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    """
    Compute basic classification metrics:
    Accuracy, Balanced Accuracy, F1, TPR, TNR.
    """
    y_true = y_true.astype(int)
    y_pred = y_pred.astype(int)

    acc = float((y_true == y_pred).mean())

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    tpr = _safe_div(tp, tp + fn)  # True Positive Rate
    tnr = _safe_div(tn, tn + fp)  # True Negative Rate
    bacc = float(np.nanmean([tpr, tnr]))

    precision = _safe_div(tp, tp + fp)
    recall = tpr
    f1 = _safe_div(2 * precision * recall, precision + recall)

    return {
        "ACC": acc,
        "BACC": bacc,
        "F1": f1,
        "TPR": tpr,
        "TNR": tnr,
    }


# =================================================
# Protected attribute handling
# =================================================
def pick_protected_column(df: pd.DataFrame) -> str:
    """
    Select a protected attribute column.
    Priority: Gender/Sex > Geography > Age.
    """
    for c in ["Gender", "Sex", "gender", "sex"]:
        if c in df.columns:
            return c
    for c in ["Geography", "Age", "geography", "age"]:
        if c in df.columns:
            return c
    raise ValueError("No protected attribute found (Gender/Sex/Geography/Age).")


def binarize_groups(series: pd.Series) -> Tuple[pd.Series, Tuple[str, str]]:
    """
    Convert a protected attribute into two groups.
    - Numeric: split by median
    - Categorical: use two most frequent categories
    """
    s = series.copy()

    if pd.api.types.is_numeric_dtype(s):
        median_val = float(np.nanmedian(s.to_numpy(dtype=float)))
        groups = (s.astype(float) >= median_val).map(
            {True: f">=median({median_val:.2f})", False: f"<median({median_val:.2f})"}
        )
        return groups, (f">=median({median_val:.2f})", f"<median({median_val:.2f})")

    s = s.astype(str)
    counts = s.value_counts(dropna=True)
    if len(counts) < 2:
        raise ValueError("Protected attribute has fewer than two groups.")

    g1, g0 = counts.index[0], counts.index[1]
    groups = s.where(s.isin([g1, g0]))
    return groups, (g1, g0)


# =================================================
# Group-level statistics and fairness metrics
# =================================================
def group_stats(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Compute group-level statistics:
    sample size, positive prediction rate, TPR, FPR, confusion counts.
    """
    rows = []
    for g, sub in df.groupby(group_col, dropna=False):
        n = int(len(sub))
        if n == 0:
            continue

        y_true = sub["y_true"].to_numpy(dtype=int)
        y_pred = sub["y_pred"].to_numpy(dtype=int)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        pos_rate = float((y_pred == 1).mean())
        tpr = _safe_div(tp, tp + fn)
        fpr = _safe_div(fp, fp + tn)

        rows.append(
            {
                "group": g,
                "n": n,
                "pos_rate": pos_rate,
                "TPR": tpr,
                "FPR": fpr,
                "TP": tp,
                "FP": fp,
                "TN": tn,
                "FN": fn,
            }
        )

    return pd.DataFrame(rows).sort_values("n", ascending=False)


def fairness_metrics(df: pd.DataFrame, protected_col: str) -> Tuple[Dict[str, object], pd.DataFrame]:
    """
    Compute group fairness metrics:
    - Statistical Parity Difference (SPD)
    - Equal Opportunity Difference (EOD)
    - Demographic Impact Ratio (DIR)
    - TPR ratio
    """
    tmp = df.copy()

    tmp["y_true"] = pd.to_numeric(tmp["y_true"], errors="coerce")
    tmp["y_pred"] = pd.to_numeric(tmp["y_pred"], errors="coerce")
    tmp = tmp.dropna(subset=["y_true", "y_pred", protected_col]).copy()

    tmp["y_true"] = tmp["y_true"].astype(int)
    tmp["y_pred"] = tmp["y_pred"].astype(int)

    groups, (g1, g0) = binarize_groups(tmp[protected_col])
    tmp["_group"] = groups
    tmp = tmp.dropna(subset=["_group"]).copy()

    gs = group_stats(tmp, "_group")

    def get_val(group: str, col: str) -> float:
        row = gs[gs["group"] == group]
        return float(row.iloc[0][col]) if not row.empty else float("nan")

    p1 = get_val(g1, "pos_rate")
    p0 = get_val(g0, "pos_rate")
    tpr1 = get_val(g1, "TPR")
    tpr0 = get_val(g0, "TPR")

    spd = p1 - p0
    eod = tpr1 - tpr0
    dir_ratio = _safe_div(p1, p0)
    tpr_ratio = _safe_div(tpr1, tpr0)

    overall = compute_basic_metrics(
        tmp["y_true"].to_numpy(int),
        tmp["y_pred"].to_numpy(int),
    )

    summary = {
        "protected_col": protected_col,
        "group1": g1,
        "group0": g0,
        "ACC": overall["ACC"],
        "BACC": overall["BACC"],
        "F1": overall["F1"],
        "SPD(P(yhat=1|g1)-P(yhat=1|g0))": spd,
        "EOD(TPR_g1-TPR_g0)": eod,
        "DIR(P(yhat=1|g1)/P(yhat=1|g0))": dir_ratio,
        "TPR_ratio(TPR_g1/TPR_g0)": tpr_ratio,
    }
# Remarks:
# - Statistical Parity Difference (SPD) measures group-level outcome disparity
#   and is commonly used to assess disparate impact in credit decisions.
# - Equal Opportunity Difference (EOD) compares True Positive Rates across
#   groups and is a standard fairness criterion in credit scoring.
# - Disparate Impact Ratio (DIR) is widely used in banking and regulatory
#   contexts to quantify relative acceptance rates between groups.
# - TPR ratio provides a normalized view of Equal Opportunity across groups.
# - These metrics follow common fairness evaluation practices in
#   credit-scoring and bank decision-making literature.
# References:
# - Hardt, M., Price, E., Srebro, N. (2016).
#   Equality of Opportunity in Supervised Learning.
# - Barocas, S., Selbst, A. D. (2016).
#   Big Data's Disparate Impact. California Law Review.
# - Kleinberg, J., Mullainathan, S., Raghavan, M. (2017).
#   Inherent Trade-Offs in the Fair Determination of Risk Scores.
# - Fuster, A., et al. (2019).
#   Predictably Unequal? The Effects of Machine Learning on Credit Markets.

    return summary, gs


def load_predictions(path: Path) -> pd.DataFrame:
    """Load predictions.csv and check required columns."""
    df = pd.read_csv(path)
    for col in ["y_true", "y_pred"]:
        if col not in df.columns:
            raise ValueError(f"{path} is missing required column: {col}")
    return df


# =================================================
# Main entry point
# =================================================
def main() -> None:
    # TODO: update these paths to your actual prediction files
    baseline_csv = Path(
        r"C:\Users\wenyi\Documents\AIDE_BANKBIAS_WORKSPACE_NAMED\OUTPUT\RUN_BASELINE\predictions_baseline.csv"
    )
    fairness_csv = Path(
        r"C:\Users\wenyi\Documents\AIDE_BANKBIAS_WORKSPACE_NAMED\OUTPUT\RUN_PROMPT\predictions_fairness.csv"
    )

    df_baseline = load_predictions(baseline_csv)
    df_fairness = load_predictions(fairness_csv)

    protected_col = pick_protected_column(df_baseline)

    sum_base, gs_base = fairness_metrics(df_baseline, protected_col)
    sum_fair, gs_fair = fairness_metrics(df_fairness, protected_col)

    comparison = pd.DataFrame(
        [
            {"method": "baseline", **sum_base},
            {"method": "fairness_prompt", **sum_fair},
        ]
    )

    pd.set_option("display.width", 180)
    pd.set_option("display.max_columns", 50)

    print("\n=== FAIRNESS COMPARISON (overall and group fairness) ===")
    print(comparison.to_string(index=False))

    print("\n=== GROUP STATISTICS: baseline ===")
    print(gs_base.to_string(index=False))

    print("\n=== GROUP STATISTICS: fairness_prompt ===")
    print(gs_fair.to_string(index=False))


if __name__ == "__main__":
    main()



=== FAIRNESS COMPARISON (overall and group fairness) ===
         method protected_col group1 group0   ACC     BACC       F1  SPD(P(yhat=1|g1)-P(yhat=1|g0))  EOD(TPR_g1-TPR_g0)  DIR(P(yhat=1|g1)/P(yhat=1|g0))  TPR_ratio(TPR_g1/TPR_g0)
       baseline        Gender   Male Female 0.808 0.576731 0.283582                       -0.068365           -0.115362                        0.323623                  0.511285
fairness_prompt        Gender   Male Female 0.797 0.575313 0.287719                       -0.080806           -0.110996                        0.352159                  0.554102

=== GROUP STATISTICS: baseline ===
 group    n  pos_rate      TPR      FPR  TP  FP  TN  FN
  Male 1070  0.032710 0.120690 0.015625  21  14 882 153
Female  930  0.101075 0.236052 0.055954  55  39 658 178

=== GROUP STATISTICS: fairness_prompt ===
 group    n  pos_rate      TPR      FPR  TP  FP  TN  FN
  Male 1070  0.043925 0.137931 0.025670  24  23 873 150
Female  930  0.124731 0.248927 0.083214  58  58 6